# UHPC Dataset


## Data preprocessing

In [283]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, MinMaxScaler
from sklearn.model_selection import KFold, train_test_split

In [284]:
df = pd.read_excel('../Datasets/UHPC_data_v2.xlsx', sheet_name=1, header=2)
df.head();
df.describe()

,Cement Amount (kg/m³),Cement Grade (Mpa),Silica Fume (kg/m³),Flayash Amount (kg/m³),Limestone Powder (kg/m3),Quartzpowder (kg/m3),Glass powder (kg/m3),Rice husk ash (kg/m3),Metakaolin (kg/m³),GGBFS (kg/m³),...,Air Content %,Air Void %,Porosity,Water absorption (%),28-day Shrinkage (µstrain),56-day Shrinkage (µstrain),Cycles,Total Charge Passed (Coulombs),Surface Resistivity (kΩ-mm),Unnamed: 90
count,2188.000000,1328.000000,2188.000000,2188.000000,2188.000000,2188.000000,2188.000000,2188.000000,2188.000000,2188.000000,...,47.000000,38.000000,239.000000,23.000000,112.000000,66.000000,22.000000,108.000000,13.000000,165.000000
mean,789.131752,48.927711,180.565814,58.466501,12.893150,57.306427,37.138388,2.759762,4.643419,41.730027,...,3.332553,4.513158,6.223389,1.325391,464.367809,358.530303,76.818182,410.327778,460.615385,2017.036364
std,199.520609,4.831452,89.350135,141.395621,61.276145,111.426777,125.614608,24.297141,30.903864,141.142663,...,1.073982,1.253788,5.167403,0.618443,412.669297,339.593613,40.056836,559.598811,67.779715,4.175847
min,170.000000,42.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.400000,2.200000,0.690000,0.430000,6.000000,25.000000,17.700000,9.000000,386.000000,2006.000000
25%,675.000000,42.500000,133.425000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.430000,3.800000,2.150000,0.825000,227.500000,238.500000,28.700000,104.625000,424.000000,2015.000000
50%,795.400000,52.500000,197.100000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,3.300000,4.400000,4.400000,1.200000,327.500000,288.000000,105.500000,142.000000,437.000000,2018.000000
75%,900.000000,52.500000,227.250000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.000000,5.200000,9.450000,1.730000,530.000000,387.500000,106.000000,421.250000,463.000000,2020.000000
max,1856.700000,53.000000,617.647059,1152.000000,1058.200000,528.000000,1067.000000,481.060000,510.000000,768.000000,...,6.300000,7.900000,25.890000,2.637000,2100.000000,2300.000000,109.000000,1970.000000,603.000000,2025.000000


In [285]:
tw_ei_da_co=df[df.columns[53]].count()
print(f'Number of rows in 28-Day = {tw_ei_da_co}')

Number of rows in 28-Day = 2073


In [286]:
tw_ei_da = df.columns[53]
df_drop=df.dropna(subset=[tw_ei_da]).copy()


In [287]:

def clean_cement_type(text):
    
    text = str(text).lower().strip()
    
    if '52.5' in text or 'p.o 52.5' in text:
        return 'OPCGrade_52_5'
        
    elif '42.5' in text or '42,5' in text:
        return 'OPCGrade_42_5'
    
    elif '53' in text:
        return 'OPCGrade_53'
    
    elif 'hs' in text:
        return 'Type_HS'
    
    elif 'i/ii' in text:
        return 'OPC_I_II'
    
    elif 'iii' in text:
        return 'OPC_III'
    
    elif 'unspecified' in text or text=='nan' or text=='':
        return 'Unspecified'
    
    elif '53' in text or 'blast' in text or 'ggbs' in text or 'white' in text or 'ii' in text:
        return 'Other'
    
    elif 'portland' in text or 'ordinary' in text or '28' in text or 'i' in text or 'opc' in text or '1' in text:
        return 'OPC_I'
    
    else:
        return 'Other'


In [288]:
cement_type = df_drop.columns[2]
# '.apply' apply function row by row
df_drop['Clean_Cement_Type'] = df_drop[cement_type].apply(clean_cement_type) 
print(df_drop['Clean_Cement_Type'].value_counts())


Clean_Cement_Type
OPCGrade_52_5    691
OPC_I            524
OPCGrade_42_5    438
Type_HS          158
OPCGrade_53       92
OPC_I_II          75
OPC_III           39
Other             37
Unspecified       19
Name: count, dtype: int64


In [289]:
# print the unique raw text labels 
#print(df_drop[df_drop['Clean_Cement_Type'] == 'Other'][cement_type].unique())
#print(df_drop[df_drop['Clean_Cement_Type'] == 'OPC_I'][cement_type].unique())

### Filling missing values


In [290]:
# filling string data
flash_ash_type = df_drop.columns[6]
slag_type = df_drop.columns[14]
sand_type = df_drop.columns[22]
fiber_type = df_drop.columns[24]
fiber_type = df_drop.columns[24]
syn_fiber_type = df_drop.columns[30]
super_plast_type = df_drop.columns[37]

df_drop[flash_ash_type] = df_drop[flash_ash_type].fillna('Unspecified')
df_drop[slag_type] = df_drop[slag_type].fillna('Unspecified')
df_drop[sand_type] = df_drop[sand_type].fillna('Unspecified')
df_drop[fiber_type] = df_drop[fiber_type].fillna('Unspecified')
df_drop[syn_fiber_type] = df_drop[syn_fiber_type].fillna('Unspecified')
df_drop[super_plast_type] = df_drop[super_plast_type].fillna('Unspecified')

In [291]:
# extracting values from the string cells
pressure_col = df_drop.columns[42] 
df_drop[pressure_col] = df_drop[pressure_col].astype(str).str.extract(r'([0-9.]+)').astype(float)

In [292]:
# filling values
index = np.r_[4,5,7:13,15:19,21,23,25:29,31:36,38]
ingredients = df_drop.columns[index]
df_drop[ingredients] = df_drop[ingredients].fillna(0.0)
#print(df_drop[ingredients].describe().loc['min'])

### Encoding (Target K-Fold)

In [293]:
#categorical_cols = ['Clean_Cement_Type', sand_type, fiber_type, syn_fiber_type, super_plast_type]

#original cement type column added instead of clean
categorical_cols = [df_drop.columns[2], sand_type, fiber_type, syn_fiber_type, super_plast_type]

y = df_drop['28-Day']

# TargetEncoder with cv
# smooth='auto' calculates the best weighted mean balance
enc = TargetEncoder(categories='auto', cv=5, smooth='auto', random_state=42)

# fit and transform the columns
enc_array = enc.fit_transform(df_drop[categorical_cols], y)

# create a clean DataFrame with the new numerical values
enc_headers = [f"{col}_encoded" for col in categorical_cols]
all_enc_df = pd.DataFrame(enc_array, columns=enc_headers, index=df_drop.index)

# combine it back with your main dataset
df_drop = pd.concat([df_drop, all_enc_df], axis=1)

In [294]:
columns_to_sc = list(ingredients) + enc_headers

for col in columns_to_sc:
    # errors='coerce' turns typos into NaN
    df_drop[col] = pd.to_numeric(df_drop[col], errors='coerce')

df_drop[columns_to_sc] = df_drop[columns_to_sc].fillna(0.0)

### Scaling

In [295]:
columns_to_sc = list(ingredients) + enc_headers

scaler = MinMaxScaler()

scaled_features = scaler.fit_transform(df_drop[columns_to_sc])

df_drop[columns_to_sc] = scaled_features

#print(df_drop[columns_to_sc].describe().loc[['min', 'max']])

In [296]:
X = df_drop[columns_to_sc]
Y = df_drop['28-Day']

In [297]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## XGBoost initialization

In [298]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error

In [299]:
model = XGBRegressor(random_state = 42)

In [300]:
model.fit(X_train,Y_train)
Y_pred = model.predict(X_test)
mse = mean_squared_error(Y_test,Y_pred)
rmse =root_mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

In [301]:
print(f'MSE {mse} RMSE {rmse} R2 {r2}')

MSE 394.5624047205207 RMSE 19.86359495963711 R2 0.7078045911461339
